<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-1-to-6/blob/main/04_Introduction_to_Neural_Learning_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predict, Compare, and Learn**

Bab 4 melengkapi dua tahap berikutnya dari paradigma utama *"predict, compare, learn"* yang diperkenalkan pada Bab 3. Tahap *compare* mengukur seberapa jauh estimasi prediksi meleset dari target. Tahap *learn* melakukan *error attribution*, yaitu proses membagi tanggung jawab eror ke masing-masing bobot agar bobot dapat diperbaiki guna menurunkan tingkat kesalahan pada iterasi berikutnya.

# **Compare: Does Your Network Make Good Predictions? &amp; Why Measure Error?**

Mengukur eror memberikan sinyal evaluasi terhadap performa jaringan. Pada bab ini, evaluasi eror menggunakan Mean Squared Error (MSE), yaitu $\text{error} = (\text{prediction} - \text{goal\_prediction})^2$. Penguadratan eror berfungsi untuk:Memastikan nilai eror selalu bernilai positif.Memberikan penalti yang jauh lebih besar pada eror skala besar dibanding eror skala kecil.

# **What’s the Simplest Form of Neural Learning? &amp; Hot and Cold Learning**

Bentuk pembelajaran paling sederhana adalah metode *Hot and Cold Learning*. Metode ini bekerja dengan menggoyangkan (*wiggling*) bobot sedikit ke atas (`weight + step_amount`) dan sedikit ke bawah (`weight - step_amount`), lalu mengevaluasi mana yang menghasilkan eror lebih kecil. Bobot kemudian digeser ke arah yang mengurangi eror tersebut secara berulang-ulang hingga eror mendekati nol.

# **Characteristics of Hot and Cold Learning**

Meskipun sederhana dan mudah diimplementasikan, *Hot and Cold Learning* memiliki beberapa kelemahan utama:
1. **Tidak Efisien**: Harus melakukan prediksi sebanyak 3 kali (prediksi awal, uji naik, uji turun) untuk setiap kali pembaruan 1 bobot.
2. **Nilai Langkah Arbitrer (** **step\_amount** **)**: Besarnya perubahan bobot ditentukan secara acak dan konstan, tidak memedulikan apakah eror saat ini masih sangat besar atau sudah sangat kecil. Hal ini menyebabkan proses pencarian bobot berjalan lambat atau berayun tanpa pernah tepat mencapai titik nol.

In [ ]:
# Kode Hot and Cold Learning untuk memprediksi goal\_prediction = 0.8

weight = 0.5
input = 0.5
goal_prediction = 0.8

step_amount = 0.001

for iteration in range(1101):
  prediction = input * weight
  error = (prediction - goal\_prediction) ** 2

  print("Error:" + str(error) + " Prediction:" + str(prediction))

  up_prediction = input * (weight + step_amount)
  up_error = (goal_prediction - up_prediction) ** 2

  down_prediction = input * (weight - step_amount)
  down_error = (goal_prediction - down_prediction) ** 2

  if (down_error < up_error):
        weight = weight - step_amount
  if (down_error > up_error):
        weight = weight + step_amount


# **Calculating Both Direction and Amount from Error**

Dibandingkan *Hot and Cold Learning*, algoritma **Gradient Descent** mampu menghitung **arah** (*direction*) dan **besaran** (*amount*) perubahan bobot secara simultan hanya dalam satu baris kode. Perhitungannya memanfaatkan turunan (*derivative*) dari fungsi eror terhadap bobot.

In [ ]:
# Menghitung arah dan jumlah perubahan dalam 1 garis kode
weight = 0.5
input = 0.5
goal_prediction = 0.8

for iteration in range(20):
    prediction = input * weight
    error = (prediction - goal_prediction) ** 2
    direction_and_amount = (prediction - goal_prediction) * input
    weight = weight - direction_and_amount
    print("Error:" + str(error) + " Prediction:" + str(prediction))

# **One Iteration of Gradient Descent**

Dalam satu iterasi *gradient descent*, terdapat variabel kunci bernama `delta`. `delta = pred - goal_pred` mewakili *raw error* (seberapa besar dan ke arah mana nodus keluaran meleset). Nilai `weight_delta = input * delta` kemudian mengukur tingkat sensitivitas eror terhadap perubahan bobot. `input` bertindak melakukan *scaling* (masukan besar memicu perubahan bobot besar), *negative reversal* (membalik arah perubahan jika input negatif), dan *stopping* (jika input 0, bobot tidak diubah).

In [ ]:
# Implementasi 1 iterasi Gradient Descent dengan delta
weight = 0.1
alpha = 0.01

def neural_network(input, weight):
    prediction = input * weight
    return prediction

number_of_toes = [8.5]
win_or_lose_binary = [24] # (won!!!)

input = number_of_toes
goal_pred = win_or_lose_binary

pred = neural_network(input, weight)
error = (pred - goal_pred) ** 2
delta = pred - goal_pred
weight_delta = input * delta
weight -= weight_delta * alpha

# **Learning Is Just Reducing Error &amp; How to Use a Derivative to Learn**

Pembelajaran pada hakikatnya adalah masalah pencarian (*search problem*) untuk meminimalkan kurva eror hingga mencapai titik terendah (titik dasar lembah). Turunan (*derivative*) mengukur kemiringan (*slope*) kurva eror pada titik bobot saat ini. Dengan menggeser bobot ke arah berlawanan dari tanda turunan, eror secara otomatis akan turun mendekati nol.

In [ ]:
# Iterasi berulang Gradient Descent menurunkan eror
weight, goal_pred, input = (0.0, 0.8, 0.5)

for iteration in range(4):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))

# **Breaking Gradient Descent &amp; Divergence**

Jika nilai variabel `input` terlalu besar (misalnya `input = 2.0`), pembaruan bobot via `weight_delta` dapat mengalami *overshooting* (melompat melewati titik minimum). Hal ini memicu fenomena **divergence**, di mana nilai prediksi dan eror bukannya mengecil melainkan meledak (*explode*) semakin besar dan berayun menjauhi target pada setiap iterasi.

In [ ]:
# Contoh saat input terlalu besar (input = 2.0) tanpa alpha -> Eror Meledak (Divergence)
weight = 0.5
goal_pred = 0.8
input = 2.0

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = input * delta
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))

# **Introducing Alpha &amp; Alpha in Code**

Untuk mencegah overkoreksi dan divergensi, diperkenalkan parameter **alpha** (atau *learning rate*). Alpha adalah faktor skala kecil (misalnya `0.1`, `0.01`, atau `0.001`) yang mengalikan `weight_delta` sebelum digunakan untuk memperbarui bobot: `weight -= alpha * weight_delta`. Parameter ini mengontrol seberapa cepat jaringan belajar tanpa mengalami lompatan ekstrem.

In [ ]:
# Solusi Divergensi: Menggunakan parameter Alpha (learning rate = 0.1)
weight = 0.5
goal_pred = 0.8
input = 2
alpha = 0.1

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    derivative = input * (pred - goal_pred)
    weight = weight - (alpha * derivative)
    print("Error:" + str(error) + " Prediction:" + str(pred))

# **Memorizing**

Penulis menekankan pentingnya menghafal dan memahami blok kode dasar *gradient descent* tunggal ini. Struktur sederhana ini akan menjadi fondasi dasar yang dipakai secara berulang saat membangun arsitektur jaringan yang lebih kompleks di bab-bab selanjutnya.